In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from ollama import Client as OllamaClient
from pymilvus import MilvusClient

# Giskard imports
from giskard.rag import KnowledgeBase, generate_testset
from giskard.llm.client import ChatMessage
from giskard.rag.question_generators import SimpleQuestionsGenerator

# =========================================================
# 1. CONFIGURATION ET IMPORTS
# =========================================================
ROOT_DIR = Path(__file__).resolve().parent.parent.parent
sys.path.append(str(ROOT_DIR))

# Chargement du .env
env_path = ROOT_DIR / ".env"
load_dotenv(env_path)

try:
    from src.utils.custom_embedding import CustomEmbedder
    print("✅ CustomEmbedder importé avec succès.")
except ImportError as e:
    print(f"❌ Erreur d'import de CustomEmbedder : {e}")
    sys.exit(1)

# Variables d'environnement
MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")
COLLECTION_NAME = os.getenv("MILVUS_COLLECTION")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
HF_EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL_NAME")

# =========================================================
# 2. WRAPPERS POUR COMPATIBILITÉ GISKARD (CORRIGÉS)
# =========================================================

class GiskardOllamaLLM:
    def __init__(self, host, model):
        self.client = OllamaClient(host=host)
        self.model = model

    # L'ajout de **kwargs est CRUCIAL pour absorber caller_id, seed, etc.
    def complete(self, messages, temperature=0.0, max_tokens=None, **kwargs):
        ollama_messages = [{"role": m.role, "content": m.content} for m in messages]
        
        try:
            resp = self.client.chat(
                model=self.model,
                messages=ollama_messages,
                options={
                    "temperature": temperature, 
                    "num_predict": max_tokens if max_tokens else 512
                }
            )
            # Gestion flexible du format de réponse Ollama
            if hasattr(resp, 'message'):
                content = resp.message.content
            else:
                content = resp['message']['content']
                
            return ChatMessage(role="assistant", content=content)
        except Exception as e:
            print(f"⚠️ Erreur Ollama : {e}")
            return ChatMessage(role="assistant", content="Erreur de génération LLM.")

class GiskardCustomEmbedder:
    def __init__(self, model_name):
        self.embedder = CustomEmbedder(model_name)

    def embed(self, texts):
        # Giskard attend un array numpy de vecteurs
        embeddings = [self.embedder(t) for t in texts]
        return np.array(embeddings)

# =========================================================
# 3. LOGIQUE PRINCIPALE
# =========================================================

def fetch_data_from_milvus(limit=150):
    print(f"🔍 Connexion à Milvus ({MILVUS_URI})...")
    client = MilvusClient(MILVUS_URI)
    
    results = client.query(
        collection_name=COLLECTION_NAME,
        filter="", 
        output_fields=["text", "source", "section_title"],
        limit=limit
    )
    
    df = pd.DataFrame(results)
    if df.empty:
        raise ValueError(f"La collection {COLLECTION_NAME} est vide !")
        
    print(f"✅ {len(df)} chunks récupérés.")
    return df

def run_testset_generation():
    # 1. Extraction
    df_raw = fetch_data_from_milvus(limit=100)

    # 2. Formatage
    df_kb = df_raw.copy()
    df_kb["document"] = (
        "Source: " + df_kb["source"].astype(str) + "\n"
        "Section: " + df_kb.get("section_title", "").fillna("").astype(str) + "\n\n"
        + df_kb["text"].astype(str)
    )

    # 3. Initialisation des composants Giskard
    llm_client = GiskardOllamaLLM(OLLAMA_URL, OLLAMA_MODEL)
    embedding_model = GiskardCustomEmbedder(HF_EMBEDDING_MODEL)

    print("🧠 Initialisation de la KnowledgeBase...")
    knowledge_base = KnowledgeBase.from_pandas(
        df_kb,
        columns=["document"],
        llm_client=llm_client,
        embedding_model=embedding_model,
    )

    print(f"✨ Génération du testset via {OLLAMA_MODEL}...")
    
    # Utilisation explicite du générateur simple pour éviter le clustering complexe
    simple_gen = SimpleQuestionsGenerator(llm_client=llm_client)

    testset = generate_testset(
        knowledge_base,
        num_questions=10, 
        language="fr",
        agent_description="Assistant technique pour le projet RAG",
        question_generators=[simple_gen]
    )

    # 4. Sauvegarde finale
    output_dir = Path("data")
    output_dir.mkdir(exist_ok=True)
    save_path = output_dir / "testset.jsonl"
    
    # Vérification si des questions ont bien été produites
    df_final = testset.to_pandas()
    if len(df_final) > 0:
        testset.save(str(save_path))
        print(f"✅ SUCCÈS ! Fichier généré : {save_path}")
        print(f"📊 Aperçu des questions :\n{df_final['question'].head()}")
    else:
        print("❌ ÉCHEC : Aucune question n'a été générée. Vérifiez les logs Ollama.")

if __name__ == "__main__":
    try:
        run_testset_generation()
    except Exception as e:
        print(f"💥 Erreur fatale : {e}")
        import traceback
        traceback.print_exc()

NameError: name '__file__' is not defined

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from ollama import Client as OllamaClient
from pymilvus import MilvusClient

# Giskard Core & LLM
from giskard.rag import KnowledgeBase, generate_testset
from giskard.llm.client import LLMClient, ChatMessage

# Giskard Generators (Indispensables pour forcer Ollama partout)
from giskard.rag.question_generators import (
    SimpleQuestionsGenerator,
    ComplexQuestionsGenerator,
    DistractingQuestionsGenerator,
    SituationalQuestionsGenerator,
    DoubleQuestionsGenerator
)

# =========================================================
# 1. CONFIGURATION ET SECURITÉ LITELLM
# =========================================================
# On définit une clé bidon pour éviter que LiteLLM ne bloque avant même d'utiliser notre client
os.environ["OPENAI_API_KEY"] = "bypass-openai-check"

ROOT_DIR = Path(__file__).resolve().parent.parent.parent
sys.path.append(str(ROOT_DIR))
load_dotenv(ROOT_DIR / ".env")

try:
    from src.utils.custom_embedding import CustomEmbedder
    print("✅ CustomEmbedder chargé.")
except ImportError:
    print("❌ Erreur import CustomEmbedder.")
    sys.exit(1)

MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")
COLLECTION_NAME = os.getenv("MILVUS_COLLECTION")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
HF_EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL_NAME")

# =========================================================
# 2. CLIENTS PERSONNALISÉS
# =========================================================

class GiskardOllamaLLM(LLMClient):
    def __init__(self, host, model):
        self.client = OllamaClient(host=host)
        self.model = model

    def complete(self, messages, temperature=0.1, max_tokens=None, **kwargs):
        ollama_messages = [{"role": m.role, "content": m.content} for m in messages]
        try:
            resp = self.client.chat(
                model=self.model,
                messages=ollama_messages,
                options={"temperature": temperature, "num_predict": max_tokens or 512}
            )
            content = resp['message']['content'] if isinstance(resp, dict) else resp.message.content
            return ChatMessage(role="assistant", content=content)
        except Exception as e:
            return ChatMessage(role="assistant", content=f"Error: {e}")

    def get_config(self):
        return {"model": self.model}

class GiskardCustomEmbedder:
    def __init__(self, model_name):
        self.embedder = CustomEmbedder(model_name)
    def embed(self, texts):
        embeddings = [self.embedder(t) for t in texts]
        return np.array(embeddings).astype(np.float32)

# =========================================================
# 3. LOGIQUE DE GÉNÉRATION
# =========================================================

def run_build_dataset():
    # A. Extraction Milvus
    print(f"🔍 Extraction Milvus ({COLLECTION_NAME})...")
    milvus = MilvusClient(MILVUS_URI)
    results = milvus.query(
        collection_name=COLLECTION_NAME,
        filter="", 
        output_fields=["text", "source"],
        limit=100 
    )
    
    if not results:
        print("❌ Milvus vide.")
        return

    df_kb = pd.DataFrame(results)
    df_kb["document"] = "Source: " + df_kb["source"] + "\n\n" + df_kb["text"]

    # B. Initialisation des composants
    llm_client = GiskardOllamaLLM(OLLAMA_URL, OLLAMA_MODEL)
    embedding_model = GiskardCustomEmbedder(HF_EMBEDDING_MODEL)

    # C. Knowledge Base
    print("🧠 Analyse thématique (UMAP/HDBSCAN)...")
    knowledge_base = KnowledgeBase.from_pandas(
        df_kb,
        columns=["document"],
        llm_client=llm_client,
        embedding_model=embedding_model,
    )

    # D. Configuration MANUELLE des générateurs pour forcer Ollama
    print("⚙️ Configuration des générateurs (Force Ollama injection)...")
    custom_generators = [
        SimpleQuestionsGenerator(llm_client=llm_client),
        ComplexQuestionsGenerator(llm_client=llm_client),
        DistractingQuestionsGenerator(llm_client=llm_client),
        SituationalQuestionsGenerator(llm_client=llm_client),
        DoubleQuestionsGenerator(llm_client=llm_client)
    ]

    # E. Génération finale
    print(f"✨ Génération du testset (10 questions) via {OLLAMA_MODEL}...")
    try:
        testset = generate_testset(
            knowledge_base,
            num_questions=10, 
            language="fr",
            agent_description="Expert RAG Ministère de l'Intérieur",
            question_generators=custom_generators  # CRITIQUE : on injecte nos générateurs
        )

        # Sauvegarde
        os.makedirs("data", exist_ok=True)
        testset.save("data/testset_complex.jsonl")
        print(f"\n✅ SUCCÈS ! Fichier généré dans data/testset_complex.jsonl")
        
    except Exception as e:
        print(f"❌ Erreur critique lors de la génération : {e}")

if __name__ == "__main__":
    run_build_dataset()

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from pathlib import Path
from dotenv import load_dotenv
from ollama import Client as OllamaClient
from pymilvus import MilvusClient

# Giskard Core
from giskard.rag import KnowledgeBase, generate_testset
from giskard.llm.client import LLMClient, ChatMessage
import giskard.llm  # Pour configurer le client par défaut

# Giskard Generators
from giskard.rag.question_generators import (
    SimpleQuestionsGenerator,
    ComplexQuestionsGenerator,
    DistractingQuestionsGenerator,
    SituationalQuestionsGenerator,
    DoubleQuestionsGenerator
)

# =========================================================
# 1. CONFIGURATION
# =========================================================
ROOT_DIR = Path(__file__).resolve().parent.parent.parent
sys.path.append(str(ROOT_DIR))
load_dotenv(ROOT_DIR / ".env")

try:
    from src.utils.custom_embedding import CustomEmbedder
    print("✅ CustomEmbedder chargé.")
except ImportError:
    print("❌ Erreur import CustomEmbedder.")
    sys.exit(1)

MILVUS_URI = os.getenv("MILVUS_URI", "http://localhost:19530")
COLLECTION_NAME = os.getenv("MILVUS_COLLECTION")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
HF_EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL_NAME")

# =========================================================
# 2. CLIENT OLLAMA PERSONNALISÉ
# =========================================================

class GiskardOllamaLLM(LLMClient):
    def __init__(self, host, model):
        self.client = OllamaClient(host=host)
        self.model = model

    def complete(self, messages, temperature=0.1, max_tokens=None, **kwargs):
        ollama_messages = [{"role": m.role, "content": m.content} for m in messages]
        try:
            resp = self.client.chat(
                model=self.model,
                messages=ollama_messages,
                options={"temperature": temperature, "num_predict": max_tokens or 512}
            )
            content = resp['message']['content'] if isinstance(resp, dict) else resp.message.content
            return ChatMessage(role="assistant", content=content)
        except Exception as e:
            print(f"⚠️ Erreur Ollama: {e}")
            return ChatMessage(role="assistant", content="Erreur de génération.")

    def get_config(self):
        return {"model": self.model, "host": "ollama"}

class GiskardCustomEmbedder:
    def __init__(self, model_name):
        self.embedder = CustomEmbedder(model_name)
    def embed(self, texts):
        embeddings = [self.embedder(t) for t in texts]
        return np.array(embeddings).astype(np.float32)

# =========================================================
# 3. LOGIQUE PRINCIPALE
# =========================================================

def run_build_dataset():
    # --- A. Extraction des données ---
    print(f"🔍 Extraction Milvus ({COLLECTION_NAME})...")
    milvus = MilvusClient(MILVUS_URI)
    results = milvus.query(collection_name=COLLECTION_NAME, filter="", output_fields=["text", "source"], limit=100)
    
    if not results:
        print("❌ Milvus vide.")
        return

    df_kb = pd.DataFrame(results)
    df_kb["document"] = "Source: " + df_kb["source"] + "\n\n" + df_kb["text"]

    # --- B. Injection CRITIQUE du client par défaut ---
    llm_client = GiskardOllamaLLM(OLLAMA_URL, OLLAMA_MODEL)
    embedding_model = GiskardCustomEmbedder(HF_EMBEDDING_MODEL)
    
    # CETTE LIGNE est celle qui empêche Giskard de chercher OpenAI
    giskard.llm.set_default_client(llm_client)
    print(f"🤖 Client Ollama ({OLLAMA_MODEL}) défini comme client global.")

    # --- C. Knowledge Base ---
    print("🧠 Analyse thématique...")
    knowledge_base = KnowledgeBase.from_pandas(
        df_kb,
        columns=["document"],
        llm_client=llm_client,
        embedding_model=embedding_model,
    )

    # --- D. Génération avec générateurs forcés ---
    custom_generators = [
        SimpleQuestionsGenerator(llm_client=llm_client),
        ComplexQuestionsGenerator(llm_client=llm_client),
        DistractingQuestionsGenerator(llm_client=llm_client),
        SituationalQuestionsGenerator(llm_client=llm_client),
        DoubleQuestionsGenerator(llm_client=llm_client)
    ]

    print(f"✨ Génération du testset (10 questions)...")
    try:
        testset = generate_testset(
            knowledge_base,
            num_questions=10, 
            language="fr",
            agent_description="Assistant expert Ministère de l'Intérieur",
            question_generators=custom_generators
        )

        os.makedirs("data", exist_ok=True)
        testset.save("data/testset_complex.jsonl")
        print(f"\n✅ SUCCÈS ! Fichier : data/testset_complex.jsonl")
        
    except Exception as e:
        print(f"❌ Erreur finale : {e}")

if __name__ == "__main__":
    run_build_dataset()